# Expert 04 — Performance Optimization


## 1. timeit Benchmarking

In [ ]:
import timeit

def benchmark(label, stmt, setup='pass', n=10000):
    t = timeit.timeit(stmt, setup=setup, number=n)
    print(f'{label:40}: {t*1000:.2f}ms ({n} runs)')

# String concatenation
benchmark('str += (loop)',    'r=""; [r.__add__(str(i)) for i in range(100)]')
benchmark('str join',         '"".join(str(i) for i in range(100))')
benchmark('str join (list)',  '"".join([str(i) for i in range(100)])')

print()

# List building
benchmark('append loop',      'r=[]; [r.append(i) for i in range(100)]')
benchmark('list comp',        '[i for i in range(100)]')
benchmark('list(range)',      'list(range(100))')

## 2. cProfile

In [ ]:
import cProfile
import pstats
import io

def slow_function():
    total = 0
    for i in range(100_000):
        total += i ** 2
    return total

def fast_function():
    return sum(i**2 for i in range(100_000))

# Profile
pr = cProfile.Profile()
pr.enable()
slow_function()
fast_function()
pr.disable()

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
ps.print_stats(10)
print(s.getvalue())

## 3. __slots__ Memory Savings

In [ ]:
import sys

class WithDict:
    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

class WithSlots:
    __slots__ = ('x', 'y', 'z')
    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

a = WithDict(1, 2, 3)
b = WithSlots(1, 2, 3)

print(f'WithDict:  {sys.getsizeof(a)} bytes + {sys.getsizeof(a.__dict__)} bytes (dict)')
print(f'WithSlots: {sys.getsizeof(b)} bytes (no dict)')

# Memory for 1 million instances
n = 1_000_000
dict_mem   = n * (sys.getsizeof(a) + sys.getsizeof(a.__dict__))
slots_mem  = n * sys.getsizeof(b)
print(f'\n1M instances:')
print(f'  WithDict:  {dict_mem / 1024**2:.1f} MB')
print(f'  WithSlots: {slots_mem / 1024**2:.1f} MB')
print(f'  Savings:   {(dict_mem - slots_mem) / 1024**2:.1f} MB')

## 4. lru_cache Performance

In [ ]:
from functools import lru_cache, cache
import time

def fib_slow(n):
    if n <= 1: return n
    return fib_slow(n-1) + fib_slow(n-2)

@cache
def fib_fast(n):
    if n <= 1: return n
    return fib_fast(n-1) + fib_fast(n-2)

# Compare
n = 35

start = time.perf_counter()
fib_slow(n)
t_slow = time.perf_counter() - start

start = time.perf_counter()
fib_fast(n)
t_fast = time.perf_counter() - start

print(f'fib_slow({n}): {t_slow*1000:.2f}ms')
print(f'fib_fast({n}): {t_fast*1000:.4f}ms')
print(f'Speedup: {t_slow/t_fast:.0f}x')
print(f'Cache info: {fib_fast.cache_info()}')

## 5. List Resize Behavior

In [ ]:
import sys

lst = []
prev_size = sys.getsizeof(lst)
print(f'len=0: {prev_size} bytes')

for i in range(25):
    lst.append(i)
    size = sys.getsizeof(lst)
    if size != prev_size:
        capacity = (size - 56) // 8  # approximate
        print(f'len={len(lst):2}: {size} bytes (capacity ~{capacity})')
        prev_size = size

## 6. Attribute Lookup Optimization

In [ ]:
import timeit
import math

data = list(range(10000))

# Slow: global lookup each iteration
def slow_sqrt():
    return [math.sqrt(x) for x in data]

# Fast: local reference
def fast_sqrt():
    sqrt = math.sqrt  # local binding
    return [sqrt(x) for x in data]

t1 = timeit.timeit(slow_sqrt, number=1000)
t2 = timeit.timeit(fast_sqrt, number=1000)
print(f'Global lookup: {t1*1000:.2f}ms')
print(f'Local binding: {t2*1000:.2f}ms')
print(f'Speedup: {t1/t2:.2f}x')

## 7. Generator vs List Memory

In [ ]:
import sys
import tracemalloc

def measure_memory(func):
    tracemalloc.start()
    result = func()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak

n = 1_000_000

list_mem = measure_memory(lambda: [x**2 for x in range(n)])
gen_mem  = measure_memory(lambda: sum(x**2 for x in range(n)))

print(f'List comprehension peak: {list_mem / 1024**2:.1f} MB')
print(f'Generator expression peak: {gen_mem / 1024:.1f} KB')
print(f'Memory ratio: {list_mem / gen_mem:.0f}x')